<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/Geospatial_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install kagglehub
!pip install kagglehub

import kagglehub
import pandas as pd
import os

# Check if Kaggle authentication is already set up; if not, prompt for kaggle.json
kaggle_dir = os.path.expanduser("~/.kaggle")
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")

if not os.path.exists(kaggle_json_path):
    print("Kaggle API key not found. Please upload your kaggle.json file.")
    print("Instructions: Go to Kaggle > Account > Create New API Token to download kaggle.json")
    from google.colab import files
    uploaded = files.upload()  # Upload kaggle.json
    if "kaggle.json" in uploaded:
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(kaggle_json_path, "wb") as f:
            f.write(uploaded["kaggle.json"])
        os.chmod(kaggle_json_path, 0o600)
        print("Kaggle API key set up successfully.")
    else:
        raise FileNotFoundError("Failed to upload kaggle.json. Please try again.")

# Download the dataset
path = kagglehub.dataset_download("gauravmalik26/food-delivery-dataset")

print("Path to dataset files:", path)

# List files in the dataset directory for verification
dataset_files = os.listdir(path)
print("Files in dataset:", dataset_files)

# Load the specific CSV file (train.csv)
file_path = os.path.join(path, "train.csv")
try:
    # Load the dataset into a pandas DataFrame
    data = pd.read_csv(file_path)
    # Print the first 5 rows to verify
    print("\nHead of the dataset (train.csv):")
    print(data.head())
    # Print DataFrame info to confirm it's ready for manipulation
    print("\nDataset Info:")
    print(data.info())
except FileNotFoundError:
    print("Error: 'train.csv' not found. Available files:", dataset_files)
    raise
except Exception as e:
    print(f"Error loading the dataset: {e}")
    raise

print(f"\nDataset has {data.shape[0]} rows and {data.shape[1]} columns.")

Path to dataset files: /kaggle/input/food-delivery-dataset
Files in dataset: ['Sample_Submission.csv', 'train.csv', 'test.csv']

Head of the dataset (train.csv):
        ID Delivery_person_ID  ...            City Time_taken(min)
0  0x4607     INDORES13DEL02   ...          Urban         (min) 24
1  0xb379     BANGRES18DEL02   ...  Metropolitian         (min) 33
2  0x5d6d     BANGRES19DEL01   ...          Urban         (min) 26
3  0x7a6a    COIMBRES13DEL02   ...  Metropolitian         (min) 21
4  0x70a2     CHENRES12DEL01   ...  Metropolitian         (min) 30

[5 rows x 20 columns]

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  object 
 1   Delivery_person_ID           45593 non-null  object 
 2   Delivery_person_Age          45593 non-null  object 

In [10]:
from sklearn.cluster import KMeans
from geopy.distance import geodesic

In [12]:
def calculate_distance(row):
    return geodesic(
        (row['Restaurant_latitude'], row['Restaurant_longitude']),
        (row['Delivery_location_latitude'], row['Delivery_location_longitude'])
    ).km

data['Distance_km'] = data.apply(calculate_distance, axis=1)

In [13]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=data['Delivery_location_longitude'],
    lat=data['Delivery_location_latitude'],
    mode='markers',
    marker=dict(color='blue', size=6, opacity=0.7),
    name='Delivery Locations',
    hovertemplate='Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra>Delivery</extra>'
))
fig.update_layout(
    title='Mapping Our Reach - Delivery Locations Across India',
    geo=dict(
        scope='asia',
        showland=True,
        landcolor='rgb(229, 229, 229)',
        showcountries=True,
        countrycolor='rgb(200, 200, 200)',
        lonaxis=dict(range=[68, 98]),
        lataxis=dict(range=[6, 38])
    ),
    margin=dict(l=0, r=0, t=60, b=0),
    showlegend=False
)
fig.show()

In [15]:
from sklearn.cluster import KMeans
x = data[['Delivery_location_latitude', 'Delivery_location_longitude']]
k = 3
kmeans = KMeans(n_clusters=k, random_state=42)
data['Cluster'] = kmeans.fit_predict(x)
centroids = kmeans.cluster_centers_

fig = go.Figure()
for cluster_label in sorted(data['Cluster'].unique()):
  cluster_data = data[data['Cluster'] == cluster_label]
  fig.add_trace(go.Scattergeo(
      lon=cluster_data['Delivery_location_longitude'],
      lat=cluster_data['Delivery_location_latitude'],
      mode='markers',
      name=f'Cluster {cluster_label}',
      marker=dict(size=6, opacity=0.7),
      hovertemplate='<b>Cluster:</b> %{text}<br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>',
      text=[f"{cluster_label}"] * len(cluster_data)
  ))

  fig.add_trace(go.Scattergeo(
      lon=centroids[:, 1],
      lat=centroids[:, 0],
      mode='markers',
      name='Centroids',
      marker=dict(size=15, symbol='x', color='red', line=dict(width=2, color='black')),
      hovertemplate='<b>Centroid</b><br>Lat: %{lat:.4f}<br>Lon: %{lon:.4f}<extra></extra>'
  ))
  fig.update_layout(
      title=f'Geo-Spatial Clustering of Delivery Locations (k = {k})',
      geo=dict(
          scope='asia',
          showland=True,
          landcolor="rgb(229, 229, 229)",
          showcountries=True,
          countrycolor="rgb(204, 204, 204)",
          lonaxis=dict(range=[68, 98]),
          lataxis=dict(range=[68, 98]),
      ),
      legend_title='Clusters',
      margin=dict(l=0, r=0, t=60, b=0)
  )
  fig.show()

In [16]:
filtered_data = data[data['Cluster'] != 1]
filtered_centroids = centroids[[0,2]]
cluster_labels = {
    0: "Central Delivery Zone",
    2: "Southern Delivery Zone"
}
filtered_data['Optimized_Zone'] = filtered_data['Cluster'].map(cluster_labels)

<ipython-input-16-d4eed990cdee>:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

